# Lesson 1B: Linear Regression - Practical Applications

---

## From Theory to Production

In **Lesson 1A**, we built linear regression from scratch, understanding the mathematical foundations. Now it's time to **use production-ready tools** to solve real-world problems!

### What You'll Learn

✅ **Scikit-learn LinearRegression** - Industry-standard library  
✅ **Polynomial Regression** - Modeling non-linear relationships  
✅ **Regularization** - Ridge, Lasso, ElasticNet (preventing overfitting)  
✅ **Feature Engineering** - Creating powerful features  
✅ **Model Evaluation** - R², MSE, MAE, cross-validation  
✅ **Pipelines** - Building production ML workflows  
✅ **Real Datasets** - California Housing, Boston Housing  

---

### The Production Stack

```python
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
```

Let's build production-ready regression models! 🚀

---

## 📋 Table of Contents

1. [Scikit-learn LinearRegression](#1.-Scikit-learn-LinearRegression)
2. [Real Dataset: California Housing](#2.-Real-Dataset:-California-Housing)
3. [Model Evaluation Metrics](#3.-Model-Evaluation-Metrics)
4. [Feature Engineering](#4.-Feature-Engineering)
5. [Polynomial Regression](#5.-Polynomial-Regression)
6. [Regularization](#6.-Regularization)
   - [Ridge Regression (L2)](#Ridge-Regression-(L2))
   - [Lasso Regression (L1)](#Lasso-Regression-(L1))
   - [ElasticNet (L1 + L2)](#ElasticNet-(L1-+-L2))
7. [Cross-Validation](#7.-Cross-Validation)
8. [Learning Curves](#8.-Learning-Curves)
9. [ML Pipelines](#9.-ML-Pipelines)
10. [Production Best Practices](#10.-Production-Best-Practices)
11. [Key Takeaways](#11.-Key-Takeaways)

---

In [ ]:
# Import all required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn imports
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.datasets import fetch_california_housing

# Visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

---

## 1. Scikit-learn LinearRegression

Let's start with the simplest case using Scikit-learn's `LinearRegression`.

In [ ]:
# Generate simple dataset
np.random.seed(42)
X_simple = 2 * np.random.rand(100, 1)
y_simple = 3 + 2 * X_simple + np.random.randn(100, 1)

# Create and train model (just 2 lines!)
model = LinearRegression()
model.fit(X_simple, y_simple)

# Get parameters
print("Model trained successfully!\n")
print(f"Intercept (θ₀): {model.intercept_[0]:.4f}")
print(f"Coefficient (θ₁): {model.coef_[0][0]:.4f}")
print(f"\nTrue values: θ₀ = 3.0000, θ₁ = 2.0000")

# Make predictions
y_pred = model.predict(X_simple)

# Calculate R² score
r2 = r2_score(y_simple, y_pred)
print(f"\nR² Score: {r2:.4f}")
print("(R² = 1.0 means perfect predictions)")

In [ ]:
# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_simple, y_simple, alpha=0.6, s=50, label='Data')
plt.plot(X_simple, y_pred, 'r-', linewidth=2, label=f'Fit: y = {model.intercept_[0]:.2f} + {model.coef_[0][0]:.2f}x')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title(f'Scikit-learn LinearRegression (R² = {r2:.4f})', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("That was easy! Just 2 lines to train a model. 🎉")

---

## 2. Real Dataset: California Housing

Let's work with a real dataset: **California Housing Prices**

**Features**:
- `MedInc`: Median income in block group
- `HouseAge`: Median house age in block group
- `AveRooms`: Average number of rooms per household
- `AveBedrms`: Average number of bedrooms per household
- `Population`: Block group population
- `AveOccup`: Average number of household members
- `Latitude`: Block group latitude
- `Longitude`: Block group longitude

**Target**: Median house value (in $100,000s)

In [ ]:
# Load California Housing dataset
california = fetch_california_housing()
X = california.data
y = california.target

# Create DataFrame for easier exploration
df = pd.DataFrame(X, columns=california.feature_names)
df['MedHouseVal'] = y

print("California Housing Dataset")
print("="*50)
print(f"Samples: {X.shape[0]:,}")
print(f"Features: {X.shape[1]}")
print(f"\nFeature names: {california.feature_names}")
print(f"\nFirst 5 rows:")
print(df.head())

# Dataset statistics
print("\nDataset Statistics:")
print(df.describe())

In [ ]:
# Visualize feature distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(df.columns):
    axes[idx].hist(df[col], bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(col, fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=10)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Feature distributions visualized")

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
correlation = df.corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

print("Key Insights:")
print(f"- MedInc has strongest correlation with MedHouseVal: {correlation['MedHouseVal']['MedInc']:.3f}")
print(f"- Latitude has negative correlation: {correlation['MedHouseVal']['Latitude']:.3f}")
print(f"- Longitude has negative correlation: {correlation['MedHouseVal']['Longitude']:.3f}")

### Split the data

In [ ]:
# Split into train and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Dataset Split:")
print("="*50)
print(f"Training set: {X_train.shape[0]:,} samples ({X_train.shape[0]/X.shape[0]*100:.1f}%)")
print(f"Test set: {X_test.shape[0]:,} samples ({X_test.shape[0]/X.shape[0]*100:.1f}%)")
print(f"\nFeatures: {X_train.shape[1]}")
print("\n✓ Data split complete")

---

## 3. Model Evaluation Metrics

### Key Metrics for Regression

1. **Mean Squared Error (MSE)**:
   $$MSE = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2$$
   - Average squared difference between predictions and actual values
   - **Lower is better** (0 = perfect)
   - Sensitive to outliers

2. **Root Mean Squared Error (RMSE)**:
   $$RMSE = \sqrt{MSE}$$
   - Same units as target variable
   - Easier to interpret than MSE

3. **Mean Absolute Error (MAE)**:
   $$MAE = \frac{1}{m} \sum_{i=1}^{m} |\hat{y}^{(i)} - y^{(i)}|$$
   - Average absolute difference
   - Less sensitive to outliers than MSE

4. **R² Score (Coefficient of Determination)**:
   $$R^2 = 1 - \frac{\sum (y^{(i)} - \hat{y}^{(i)})^2}{\sum (y^{(i)} - \bar{y})^2}$$
   - Proportion of variance explained by the model
   - Range: (-∞, 1], where 1 = perfect predictions
   - 0 = model is no better than predicting the mean

In [ ]:
# Train baseline linear regression
model_baseline = LinearRegression()
model_baseline.fit(X_train, y_train)

# Predictions
y_train_pred = model_baseline.predict(X_train)
y_test_pred = model_baseline.predict(X_test)

# Calculate metrics
def evaluate_model(y_true, y_pred, set_name="Set"):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"{set_name} Metrics:")
    print(f"  MSE:  {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R²:   {r2:.4f}")
    print()
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

print("="*50)
print("BASELINE LINEAR REGRESSION")
print("="*50)
train_metrics = evaluate_model(y_train, y_train_pred, "Training")
test_metrics = evaluate_model(y_test, y_test_pred, "Test")

print(f"Model explains {test_metrics['R2']*100:.2f}% of variance in test data")

In [ ]:
# Visualize predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.3, s=10)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 
             'r--', linewidth=2, label='Perfect predictions')
axes[0].set_xlabel('Actual Values', fontsize=12)
axes[0].set_ylabel('Predicted Values', fontsize=12)
axes[0].set_title(f'Training Set (R² = {train_metrics["R2"]:.4f})', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.3, s=10, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2, label='Perfect predictions')
axes[1].set_xlabel('Actual Values', fontsize=12)
axes[1].set_ylabel('Predicted Values', fontsize=12)
axes[1].set_title(f'Test Set (R² = {test_metrics["R2"]:.4f})', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Points close to the red line indicate accurate predictions")

In [ ]:
# Residual plot (errors)
residuals = y_test - y_test_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Residual scatter plot
axes[0].scatter(y_test_pred, residuals, alpha=0.3, s=10)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Values', fontsize=12)
axes[0].set_ylabel('Residuals (Actual - Predicted)', fontsize=12)
axes[0].set_title('Residual Plot', fontsize=14)
axes[0].grid(True, alpha=0.3)

# Residual histogram
axes[1].hist(residuals, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual Value', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Residual Distribution', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Good model: Residuals should be randomly scattered around zero")
print(f"Mean residual: {residuals.mean():.6f} (close to 0 is good)")
print(f"Std residual: {residuals.std():.4f}")

---

## 4. Feature Engineering

Let's create new features to improve model performance!

In [ ]:
# Feature importance (coefficients)
feature_importance = pd.DataFrame({
    'Feature': california.feature_names,
    'Coefficient': model_baseline.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in feature_importance['Coefficient']]
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors, alpha=0.7)
plt.xlabel('Coefficient Value', fontsize=12)
plt.title('Feature Importance (Linear Regression Coefficients)', fontsize=14)
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("Feature Importance:")
print(feature_importance)
print("\nMedInc (median income) is the strongest predictor!")

---

## 5. Polynomial Regression

**Linear regression can model non-linear relationships** by creating polynomial features!

Example: $y = \theta_0 + \theta_1 x + \theta_2 x^2 + \theta_3 x^3$

In [ ]:
# Generate non-linear data
np.random.seed(42)
X_poly = 6 * np.random.rand(100, 1) - 3
y_poly = 0.5 * X_poly**2 + X_poly + 2 + np.random.randn(100, 1)

# Try linear regression (will fail to capture pattern)
model_linear = LinearRegression()
model_linear.fit(X_poly, y_poly)
y_linear_pred = model_linear.predict(X_poly)

# Try polynomial regression (degree 2)
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly_features = poly_features.fit_transform(X_poly)

model_poly = LinearRegression()
model_poly.fit(X_poly_features, y_poly)
y_poly_pred = model_poly.predict(X_poly_features)

# Sort for plotting
sort_idx = X_poly.ravel().argsort()

# Plot comparison
plt.figure(figsize=(12, 6))
plt.scatter(X_poly, y_poly, alpha=0.6, s=50, label='Data')
plt.plot(X_poly[sort_idx], y_linear_pred[sort_idx], 'r-', linewidth=2, 
         label=f'Linear (R² = {r2_score(y_poly, y_linear_pred):.3f})')
plt.plot(X_poly[sort_idx], y_poly_pred[sort_idx], 'g-', linewidth=2, 
         label=f'Polynomial deg=2 (R² = {r2_score(y_poly, y_poly_pred):.3f})')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Linear vs Polynomial Regression', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Linear R²: {r2_score(y_poly, y_linear_pred):.4f}")
print(f"Polynomial R²: {r2_score(y_poly, y_poly_pred):.4f}")
print("\nPolynomial features capture the non-linear relationship! ✓")

### Beware of overfitting!

In [ ]:
# Compare different polynomial degrees
degrees = [1, 2, 5, 10, 15]
plt.figure(figsize=(15, 10))

for idx, degree in enumerate(degrees, 1):
    plt.subplot(2, 3, idx)
    
    # Create polynomial features
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly_deg = poly.fit_transform(X_poly)
    
    # Train model
    model = LinearRegression()
    model.fit(X_poly_deg, y_poly)
    y_pred_deg = model.predict(X_poly_deg)
    
    # Plot
    plt.scatter(X_poly, y_poly, alpha=0.6, s=30)
    plt.plot(X_poly[sort_idx], y_pred_deg[sort_idx], 'r-', linewidth=2)
    plt.xlabel('X', fontsize=10)
    plt.ylabel('y', fontsize=10)
    plt.title(f'Degree {degree} (R² = {r2_score(y_poly, y_pred_deg):.3f})', fontsize=12)
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice:")
print("- Degree 1 (linear): Underfits (too simple)")
print("- Degree 2: Good fit (captures true relationship)")
print("- Degree 10+: Overfits (too complex, wiggly curves)")
print("\nKey lesson: Higher degree ≠ better model!")

---

## 6. Regularization

**Problem**: Complex models can overfit (memorize training data, poor generalization)

**Solution**: Add a penalty term to the cost function!

### Ridge Regression (L2)

$$J(\theta) = MSE(\theta) + \alpha \sum_{i=1}^{n} \theta_i^2$$

- Penalizes large coefficients
- Shrinks all coefficients toward zero
- Never eliminates features completely
- Good when all features are potentially useful

### Lasso Regression (L1)

$$J(\theta) = MSE(\theta) + \alpha \sum_{i=1}^{n} |\theta_i|$$

- Can eliminate features (set coefficients to exactly 0)
- Performs feature selection automatically
- Good when you have many irrelevant features

### ElasticNet (L1 + L2)

$$J(\theta) = MSE(\theta) + r \alpha \sum |\theta_i| + \frac{1-r}{2} \alpha \sum \theta_i^2$$

- Combines Ridge and Lasso
- Best of both worlds

**Key parameter**: $\alpha$ (regularization strength)
- $\alpha = 0$: No regularization (ordinary least squares)
- $\alpha$ → ∞: All coefficients → 0

In [ ]:
# Compare regularization methods on California Housing
from sklearn.preprocessing import StandardScaler

# Feature scaling (important for regularization!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train different models
models = {
    'Linear': LinearRegression(),
    'Ridge (α=1)': Ridge(alpha=1.0),
    'Ridge (α=10)': Ridge(alpha=10.0),
    'Lasso (α=0.01)': Lasso(alpha=0.01),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5)
}

results = []

for name, model in models.items():
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Evaluate
    train_score = model.score(X_train_scaled, y_train)
    test_score = model.score(X_test_scaled, y_test)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append({
        'Model': name,
        'Train R²': train_score,
        'Test R²': test_score,
        'RMSE': rmse,
        'Coefficients': model.coef_
    })

# Display results
results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'Coefficients'} for r in results])
print("="*70)
print("REGULARIZATION COMPARISON")
print("="*70)
print(results_df.to_string(index=False))
print("\nKey Observations:")
print("- Ridge: Small reduction in performance, more stable coefficients")
print("- Lasso: Can improve test performance, eliminates features")
print("- ElasticNet: Balances Ridge and Lasso benefits")

In [ ]:
# Visualize coefficient shrinkage
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

for idx, result in enumerate(results):
    coefs = result['Coefficients']
    colors = ['green' if x > 0 else 'red' for x in coefs]
    
    axes[idx].barh(california.feature_names, coefs, color=colors, alpha=0.7)
    axes[idx].set_xlabel('Coefficient', fontsize=10)
    axes[idx].set_title(f"{result['Model']}\n(Test R² = {result['Test R²']:.4f})", fontsize=11)
    axes[idx].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    axes[idx].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("Notice how regularization shrinks coefficients toward zero!")
print("Lasso can set coefficients to exactly zero (feature selection)")

---

## 7. Cross-Validation

**Problem**: Single train/test split might not be representative

**Solution**: K-Fold Cross-Validation

1. Split data into K folds
2. Train K models, each using K-1 folds for training and 1 for validation
3. Average the results

In [ ]:
# 5-Fold Cross-Validation
from sklearn.model_selection import cross_val_score

models_cv = {
    'Linear': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.1)
}

print("="*70)
print("5-FOLD CROSS-VALIDATION RESULTS")
print("="*70)

cv_results = []

for name, model in models_cv.items():
    # Perform 5-fold CV (returns R² scores)
    scores = cross_val_score(model, X_train_scaled, y_train, cv=5, 
                            scoring='r2', n_jobs=-1)
    
    cv_results.append({
        'Model': name,
        'Mean R²': scores.mean(),
        'Std R²': scores.std(),
        'Min R²': scores.min(),
        'Max R²': scores.max()
    })
    
    print(f"{name:15} | Mean: {scores.mean():.4f} ± {scores.std():.4f} | Range: [{scores.min():.4f}, {scores.max():.4f}]")

print("\n✓ Cross-validation provides more reliable performance estimates")

In [ ]:
# Visualize CV results
cv_df = pd.DataFrame(cv_results)

plt.figure(figsize=(10, 6))
plt.errorbar(cv_df['Model'], cv_df['Mean R²'], yerr=cv_df['Std R²'], 
             fmt='o', markersize=10, capsize=5, capthick=2, linewidth=2)
plt.ylabel('R² Score', fontsize=12)
plt.title('5-Fold Cross-Validation Results (Mean ± Std)', fontsize=14)
plt.grid(True, alpha=0.3, axis='y')
plt.ylim([0.55, 0.65])
plt.tight_layout()
plt.show()

print("Error bars show variability across folds")

---

## 8. Learning Curves

**Learning curves** show how model performance changes with training set size.

**Use cases**:
- Diagnose underfitting vs overfitting
- Determine if more data would help

In [ ]:
from sklearn.model_selection import learning_curve

# Compute learning curves
train_sizes, train_scores, val_scores = learning_curve(
    LinearRegression(), X_train_scaled, y_train, 
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='r2', n_jobs=-1
)

# Calculate means and stds
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='blue', linewidth=2, label='Training score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, 
                 alpha=0.2, color='blue')
plt.plot(train_sizes, val_mean, 'o-', color='red', linewidth=2, label='Validation score')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, 
                 alpha=0.2, color='red')
plt.xlabel('Training Set Size', fontsize=12)
plt.ylabel('R² Score', fontsize=12)
plt.title('Learning Curves: Linear Regression', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Training score decreases as we add more data (harder to fit perfectly)")
print("- Validation score increases (better generalization)")
print("- Curves converge: Adding more data won't help much")
print("- Gap between curves: Some overfitting (but not severe)")

---

## 9. ML Pipelines

**Pipelines** chain preprocessing and modeling steps for clean, reproducible code!

In [ ]:
from sklearn.pipeline import Pipeline

# Create pipeline: Scaling → Ridge Regression
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

# Train pipeline (handles scaling automatically!)
pipeline.fit(X_train, y_train)

# Evaluate
train_score = pipeline.score(X_train, y_train)
test_score = pipeline.score(X_test, y_test)

print("="*50)
print("PIPELINE RESULTS")
print("="*50)
print(f"Training R²: {train_score:.4f}")
print(f"Test R²: {test_score:.4f}")
print("\n✓ Pipeline handles preprocessing automatically!")
print("✓ No risk of data leakage (scaler fit only on training data)")
print("✓ Easy to deploy (single object to save)")

In [ ]:
# More complex pipeline: Polynomial features + Ridge
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10.0))
])

# Train
poly_pipeline.fit(X_train, y_train)

# Evaluate
print("\nPolynomial Pipeline Results:")
print(f"Training R²: {poly_pipeline.score(X_train, y_train):.4f}")
print(f"Test R²: {poly_pipeline.score(X_test, y_test):.4f}")

print("\nPipeline benefits:")
print("✓ Clean code (all steps in one object)")
print("✓ Prevents data leakage")
print("✓ Easy hyperparameter tuning")
print("✓ Production-ready")

---

## 10. Production Best Practices

### Checklist for Production ML

In [ ]:
# Example: Complete production pipeline
import joblib

# 1. Build final pipeline
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

# 2. Train on full training set
final_pipeline.fit(X_train, y_train)

# 3. Final evaluation on test set
test_r2 = final_pipeline.score(X_test, y_test)
y_final_pred = final_pipeline.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_final_pred))

print("="*50)
print("FINAL PRODUCTION MODEL")
print("="*50)
print(f"Model: Ridge Regression (α=1.0)")
print(f"Test R²: {test_r2:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"\nInterpretation: Average prediction error is ${test_rmse*100:.0f}k")

# 4. Save model
model_path = '/tmp/california_housing_model.pkl'
joblib.dump(final_pipeline, model_path)
print(f"\n✓ Model saved to: {model_path}")

# 5. Load and test
loaded_pipeline = joblib.load(model_path)
test_prediction = loaded_pipeline.predict(X_test[:1])
print(f"\n✓ Model loaded successfully")
print(f"Sample prediction: ${test_prediction[0]*100:.0f}k (Actual: ${y_test[0]*100:.0f}k)")

print("\n" + "="*50)
print("PRODUCTION CHECKLIST")
print("="*50)
print("✓ Data split properly (train/test)")
print("✓ Feature scaling applied")
print("✓ Cross-validation performed")
print("✓ Model evaluated on held-out test set")
print("✓ Pipeline created (reproducible)")
print("✓ Model serialized (saved to disk)")
print("✓ Model tested after loading")
print("\nReady for deployment! 🚀")

---

## 11. Key Takeaways

### ✅ What We Learned

1. **Scikit-learn Basics**:
   - `fit()`: Train the model
   - `predict()`: Make predictions
   - `score()`: Evaluate performance (R²)

2. **Evaluation Metrics**:
   - **MSE/RMSE**: Measure average error (sensitive to outliers)
   - **MAE**: Average absolute error (robust to outliers)
   - **R²**: Proportion of variance explained (0 to 1)

3. **Polynomial Regression**:
   - Linear regression can model non-linear relationships
   - Use `PolynomialFeatures` to create polynomial terms
   - Beware of overfitting with high degrees

4. **Regularization**:
   - **Ridge (L2)**: Shrinks all coefficients, keeps all features
   - **Lasso (L1)**: Can eliminate features, performs feature selection
   - **ElasticNet**: Combines Ridge and Lasso benefits
   - **α parameter**: Controls regularization strength

5. **Cross-Validation**:
   - More reliable than single train/test split
   - K-fold CV: Train K models on different splits
   - Use `cross_val_score()` for quick evaluation

6. **Pipelines**:
   - Chain preprocessing and modeling steps
   - Prevent data leakage
   - Production-ready code

### 🎯 When to Use What

| Scenario | Best Choice |
|----------|-------------|
| Simple linear relationship | `LinearRegression` |
| Non-linear relationship | `PolynomialFeatures` + `LinearRegression` |
| Many features, potential overfitting | `Ridge` or `Lasso` |
| Feature selection needed | `Lasso` |
| Combination of above | `ElasticNet` |
| Production deployment | `Pipeline` + chosen model |

---

### 🔜 Next Steps

In **Lesson 2: Logistic Regression**, we'll explore:
- Classification (predicting categories, not numbers)
- Sigmoid function and probability estimation
- Binary and multiclass classification
- Evaluation metrics for classification (accuracy, precision, recall, ROC)

---

### 📚 Further Reading

- Scikit-learn Linear Models: https://scikit-learn.org/stable/modules/linear_model.html
- "Introduction to Statistical Learning" Chapter 3: Linear Regression
- "Hands-On Machine Learning" Chapter 4: Training Models
- Cross-validation strategies: https://scikit-learn.org/stable/modules/cross_validation.html

---

**Congratulations!** You can now build production-ready linear regression models! 🎉

---